In [1]:
# ==============================================================================
# SCRIPT HUẤN LUYỆN XLM-RoBERTa (KIỂM TRA LẬP LUẬN - ARGKP) CHO KAIKO
# Dành cho Kaggle hoặc Google Colab
# ==============================================================================
# Hướng dẫn sử dụng:
# 1. Tạo một Notebook mới trên Kaggle (hoặc Google Colab).
# 2. Bật GPU (Kaggle: Settings -> Accelerator -> GPU T4 x2).
# 3. Upload file `ArgKP_combined_vi.csv` lên Kaggle.
# 4. Copy toàn bộ đoạn code này dán vào 1 cell và chạy.
# ==============================================================================

# QUAN TRỌNG: pin transformers 4.46 để tránh bug nạp trọng số (LayerNorm bị mất)
# của transformers 5.x khiến backbone XLM-R bị khởi tạo ngẫu nhiên -> model không học.
!pip install -q "transformers==4.46.3" "accelerate>=0.34.0" datasets scikit-learn pandas

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # tắt cảnh báo fork của tokenizers

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report
)

SEED = 42
set_seed(SEED)

print("Đang chuẩn bị dữ liệu...")

# 1. Load Dataset
df = pd.read_csv('/kaggle/input/datasets/phctmtjj/argkp-combined-vi/ArgKP_combined_vi.csv')

# Lọc bỏ các dòng bị lỗi dịch (rỗng)
df = df.dropna(subset=['argument_vi', 'key_point', 'label'])

# Đảm bảo label là kiểu integer (0 hoặc 1)
df['label'] = df['label'].astype(int)

print(f"Tổng số dữ liệu hợp lệ: {len(df)}")
print(df['label'].value_counts())

# Tách Train / Validation / Test = 70% / 10% / 20%
# - Validation dùng để chọn best model VÀ dò ngưỡng quyết định (KHÔNG đụng vào Test).
# - Test chỉ dùng 1 lần cuối để báo cáo -> số liệu trung thực, không rò rỉ.
trainval_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['label'])
train_df, val_df = train_test_split(
    trainval_df, test_size=0.125, random_state=SEED, stratify=trainval_df['label']
)  # 0.125 * 0.8 = 0.10 tổng thể
print(f"Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")

# Tạo HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['argument_vi', 'key_point', 'label']], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[['argument_vi', 'key_point', 'label']], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['argument_vi', 'key_point', 'label']], preserve_index=False)

# 2. Khởi tạo Tokenizer của XLM-RoBERTa (hỗ trợ đa ngôn ngữ Việt + Anh)
MODEL_NAME = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenize theo cặp câu (Sentence Pair Classification)
# Cấu trúc: <s> argument_vi </s></s> key_point </s>
def tokenize_function(examples):
    return tokenizer(
        examples["argument_vi"],
        examples["key_point"],
        truncation=True,
        max_length=256,
    )

print("Đang tokenize...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# 3. Khởi tạo Model + KIỂM TRA nạp trọng số có đúng không
model, loading_info = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # Binary Classification (1: Match, 0: Not Match)
    id2label={0: "KHONG_KHOP", 1: "KHOP"},
    label2id={"KHONG_KHOP": 0, "KHOP": 1},
    output_loading_info=True,
)

# CHỐT CHẶN AN TOÀN: chỉ được phép thiếu các key của head phân loại (classifier.*).
missing = loading_info.get("missing_keys", [])
bad = [k for k in missing if not k.startswith("classifier.")]
print(f"Missing keys (chỉ nên là classifier.*): {missing}")
if bad:
    raise RuntimeError(
        "❌ Backbone XLM-R bị thiếu trọng số (khởi tạo ngẫu nhiên): "
        f"{bad[:5]} ... -> hãy kiểm tra lại phiên bản transformers (pin 4.46.3). "
        "KHÔNG train tiếp vì model sẽ không học được."
    )
print("✅ Backbone nạp OK, chỉ head phân loại được khởi tạo mới.")

# 4. Xử lý mất cân bằng nhãn bằng Weighted Loss (tính trên tập TRAIN)
total = len(train_df)
count_0 = (train_df['label'] == 0).sum()
count_1 = (train_df['label'] == 1).sum()
weight_0 = total / (2 * count_0)
weight_1 = total / (2 * count_1)
class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float32)
print(f"Class weights: 0={weight_0:.3f}, 1={weight_1:.3f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# 5. Metrics (dùng cho chọn best model theo F1 lớp dương ở ngưỡng mặc định 0.5)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
    }

# 6. Cấu hình Training
training_args = TrainingArguments(
    output_dir="./kaiko-argkp-model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,       # chọn best model theo VALIDATION, không phải test
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("BẮT ĐẦU HUẤN LUYỆN (TRAINING)...")
trainer.train()

# 7. DÒ NGƯỠNG QUYẾT ĐỊNH trên tập VALIDATION
# Mặc định argmax = ngưỡng 0.5. Vì model thường "dè dặt" gán KHOP (recall thấp),
# ta chọn ngưỡng cho lớp KHOP tối đa hoá F1 -> đẩy Recall & F1 lên, chốt trên VAL.
def softmax_np(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

val_pred = trainer.predict(tokenized_val)
val_prob_khop = softmax_np(val_pred.predictions)[:, 1]
val_true = val_pred.label_ids

print("\n===== DÒ NGƯỠNG (trên VALIDATION) =====")
print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'f1':>8}")
best_thr, best_f1 = 0.5, -1.0
for thr in np.arange(0.30, 0.71, 0.05):
    pred = (val_prob_khop >= thr).astype(int)
    p = precision_score(val_true, pred, zero_division=0)
    r = recall_score(val_true, pred, zero_division=0)
    f = f1_score(val_true, pred, zero_division=0)
    print(f"{thr:>5.2f} {p:>10.4f} {r:>8.4f} {f:>8.4f}")
    if f > best_f1:
        best_f1, best_thr = f, thr
print(f"=> Ngưỡng tốt nhất theo F1 (VAL): {best_thr:.2f} (F1={best_f1:.4f})")

# 8. BÁO CÁO CUỐI trên TEST tại ngưỡng đã chọn (chỉ chạm test 1 lần)
test_pred = trainer.predict(tokenized_test)
test_prob_khop = softmax_np(test_pred.predictions)[:, 1]
test_true = test_pred.label_ids
y_pred_test = (test_prob_khop >= best_thr).astype(int)

print(f"\n===== BÁO CÁO CHI TIẾT (TEST @ ngưỡng {best_thr:.2f}) =====")
print(classification_report(
    test_true, y_pred_test, target_names=["KHONG_KHOP", "KHOP"], digits=4, zero_division=0
))

# 9. Lưu model + ngưỡng đã chọn (backend đọc threshold này khi suy luận)
import json
print("Đang lưu model...")
trainer.save_model("./kaiko_argkp_model_final")
tokenizer.save_pretrained("./kaiko_argkp_model_final")
with open("./kaiko_argkp_model_final/decision_threshold.json", "w") as f:
    json.dump({"khop_threshold": float(best_thr)}, f)
print(f"Đã lưu ngưỡng KHOP = {best_thr:.2f} vào decision_threshold.json")

print("HOÀN THÀNH! Bạn có thể tải thư mục ./kaiko_argkp_model_final về máy.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 64.5 MB/s eta 0:00:00


2026-07-08 16:40:16.758681: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783528817.171429      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783528817.298298      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783528818.322216      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783528818.322254      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783528818.322256      24 computation_placer.cc:177] computation placer alr

Đang chuẩn bị dữ liệu...
Tổng số dữ liệu hợp lệ: 1990
label
0    1447
1     543
Name: count, dtype: int64
Train=1393  Val=199  Test=398


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Missing keys (chỉ nên là classifier.*): ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
✅ Backbone nạp OK, chỉ head phân loại được khởi tạo mới.
Class weights: 0=0.688, 1=1.833
BẮT ĐẦU HUẤN LUYỆN (TRAINING)...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.700800,0.691055,0.331658,0.443515,0.286486,0.981481
2,0.666100,0.614582,0.783920,0.494118,0.677419,0.388889
3,0.556300,0.526821,0.773869,0.661654,0.556962,0.814815
4,0.516700,0.494386,0.814070,0.633663,0.680851,0.592593
5,0.441800,0.415372,0.829146,0.721311,0.647059,0.814815
6,0.360400,0.358112,0.859296,0.762712,0.703125,0.833333
7,0.286200,0.358463,0.874372,0.778761,0.745763,0.814815
8,0.224500,0.350764,0.889447,0.807018,0.766667,0.851852
9,0.217600,0.363660,0.879397,0.789474,0.750000,0.833333
10,0.159300,0.384810,0.889447,0.800000,0.785714,0.814815



===== DÒ NGƯỠNG (trên VALIDATION) =====
  thr  precision   recall       f1
 0.30     0.7231   0.8704   0.7899
 0.35     0.7460   0.8704   0.8034
 0.40     0.7541   0.8519   0.8000
 0.45     0.7541   0.8519   0.8000
 0.50     0.7667   0.8519   0.8070
 0.55     0.7931   0.8519   0.8214
 0.60     0.8070   0.8519   0.8288
 0.65     0.8036   0.8333   0.8182
 0.70     0.8000   0.8148   0.8073
=> Ngưỡng tốt nhất theo F1 (VAL): 0.60 (F1=0.8288)



===== BÁO CÁO CHI TIẾT (TEST @ ngưỡng 0.60) =====
              precision    recall  f1-score   support

  KHONG_KHOP     0.8997    0.9308    0.9150       289
        KHOP     0.7980    0.7248    0.7596       109

    accuracy                         0.8744       398
   macro avg     0.8488    0.8278    0.8373       398
weighted avg     0.8718    0.8744    0.8724       398

Đang lưu model...
Đã lưu ngưỡng KHOP = 0.60 vào decision_threshold.json
HOÀN THÀNH! Bạn có thể tải thư mục ./kaiko_argkp_model_final về máy.


In [2]:
import shutil
shutil.make_archive('kaiko_argkp_model_final', 'zip', './kaiko_argkp_model_final')
print("Đã nén xong! Tải file kaiko_argkp_model_final.zip về máy từ tab Output.")

Đã nén xong! Tải file kaiko_argkp_model_final.zip về máy từ tab Output.
